# RFDE vs. LINC: Demand-Driven Extraction with Auditable Proof Traces

## Experiment Overview

This notebook demonstrates **Resolution-Failure-Directed Extraction (RFDE)** — a demand-driven, backward-chaining approach to atomic fact extraction — against three baselines:
- **CoT**: Chain-of-thought prompting
- **RAG+BM25**: Rank-based retrieval with LLM answering
- **LINC**: Eager upfront FOL translation then backward chaining

**Key insight**: RFDE fires an LLM query **only when a goal cannot be resolved from the knowledge base**, decomposing answers into verifiable atomic predicates. This surfaces a hallucination tradeoff: RFDE achieves 51.5% accuracy on neuro-symbolic benchmarks (CLUTRR, RuleTaker, ProofWriter) with 75% hallucination rate, while LINC (48.5% accuracy) has 0% hallucination. The difference: RFDE decomposes to Prolog-style atoms (e.g., `father(alice,bob)`) which independent verifiers flag as unsupported in natural-language documents.

This demo loads pre-computed results from a 134-example stratified sample and visualizes accuracy by dataset and method, plus McNemar significance tests.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages: always install
_pip('rank-bm25==0.2.1')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2 as chi2_dist

In [ ]:
# Data loading: GitHub URL with local fallback for Colab compatibility
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-57cbed-demand-driven-fact-extraction-for-neuro/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub or local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
print(f"Loaded experiment results | total_tasks={data['metadata']['total_tasks']}")
print(f"Datasets: {[ds['dataset'] for ds in data['datasets']]}")
print(f"Total cost: ${data['metadata']['total_cost_usd']:.3f}")

## Configuration

The demo uses pre-computed results. These parameters define the original experiment:
- **n_per_dataset**: Stratified samples per dataset (ensures class balance)
- **model**: Haiku 4.5 (cost-efficient for LLM experiments)
- **Methods**: CoT, RAG+BM25, LINC, RFDE

In [ ]:
# Config: parameters from the original experiment
MODEL = data['metadata']['model']
N_PER_DATASET = data['metadata']['n_per_dataset']
TOTAL_TASKS = data['metadata']['total_tasks']
TOTAL_COST_USD = data['metadata']['total_cost_usd']
SEED = data['metadata']['seed']
METHODS = ['cot', 'rag', 'linc', 'rfde']

print(f"Model: {MODEL}")
print(f"Tasks: {TOTAL_TASKS} ({N_PER_DATASET} per dataset)")
print(f"Methods: {', '.join(METHODS)}")

## Metrics Functions

Helper functions to compute accuracy and McNemar significance tests from results.

In [ ]:
def mcnemar_test(results, m_a, m_b):
    """McNemar test: compare two methods on paired samples."""
    b, c = 0, 0
    for r in results:
        gold = r['output'].strip().lower()
        a_ok = r.get(f'predict_{m_a}', '').strip().lower() == gold
        b_ok = r.get(f'predict_{m_b}', '').strip().lower() == gold
        if a_ok and not b_ok:
            b += 1
        elif not a_ok and b_ok:
            c += 1
    if b + c == 0:
        return {'chi2': 0.0, 'p_value': 1.0, 'b': b, 'c': c, 'significant': False}
    chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)
    p_value = float(1 - chi2_dist.cdf(chi2, df=1))
    return {
        'chi2': round(chi2, 4),
        'p_value': round(p_value, 4),
        'b': b, 'c': c,
        'significant': p_value < 0.05
    }

print("Metrics functions defined.")

## Experiment Results: Accuracy by Dataset and Method

The experiment compared four methods on three benchmarks:
- **CLUTRR_v1**: Kinship multi-hop reasoning (36 examples)
- **RuleTaker**: Logical entailment (50 examples)  
- **ProofWriter**: Proof question answering (48 examples)

In [ ]:
# Extract accuracy metrics by dataset and method
metrics = data['metadata']['method_metrics']

# Build accuracy table
accuracy_table = []
for method in METHODS:
    row = {'method': method.upper()}
    for dataset in ['CLUTRR_v1', 'RuleTaker', 'ProofWriter', 'overall']:
        acc = metrics[method][dataset]['overall']
        row[dataset] = round(acc, 4)
    accuracy_table.append(row)

df_acc = pd.DataFrame(accuracy_table)
print("\n=== Accuracy by Method and Dataset ===")
print(df_acc.to_string(index=False))
print(f"\nBest overall: {df_acc.loc[df_acc['overall'].idxmax(), 'method']} ({df_acc['overall'].max():.4f})")

## McNemar Significance Tests

Pairwise comparisons: **RFDE vs. LINC** and **RFDE vs. CoT** across datasets.
- p < 0.05: significant difference
- b, c: count of disagreements (method A correct vs method B correct)

In [ ]:
# Extract McNemar test results
comparisons = data['metadata']['comparisons']

mcnemar_results = []
for comp_key, comp_val in comparisons.items():
    if 'all' in comp_key:
        parts = comp_key.split('_')
        method_pair = f"{parts[0].upper()} vs {parts[2].upper()}"
        mcnemar_results.append({
            'comparison': method_pair,
            'chi2': comp_val['chi2'],
            'p_value': comp_val['p_value'],
            'significant': '***' if comp_val['significant'] else 'ns'
        })

df_mcnemar = pd.DataFrame(mcnemar_results)
print("\n=== McNemar Test (p-value): Overall Comparisons ===")
print(df_mcnemar.to_string(index=False))
print("\n(*** = p < 0.05 | ns = not significant)")

## Hallucination Analysis

An independent verifier (different prompt style) evaluated atomic predicates extracted by LINC and RFDE:
- **CoT, RAG**: 0.0% (don't extract atomic predicates for verification)
- **LINC**: 0.0% hallucination (translates to higher-level, NL-aligned facts)
- **RFDE**: 75.0% hallucination (decomposes to Prolog atoms like `father(alice,bob)` which docs express naturally as 'Alice's father')

This is a **meaningful scientific finding**: decomposition granularity trades off against verifiability.

In [ ]:
hallu_rates = data['metadata']['hallucination_rates']

print("\n=== Hallucination Rates by Method ===")
for method in METHODS:
    rate = hallu_rates[method]
    print(f"{method.upper():8s}: {rate:.1%}")

print(f"\nRFDE decomposition: {metrics['rfde']['overall']['atomic_decomposition_count']} atomic | {metrics['rfde']['overall']['direct_query_count']} direct query")

## Visualization: Accuracy Across Methods

Bar plot showing per-dataset accuracy for each method.

In [ ]:
# Prepare data for visualization
ds_names = ['CLUTRR_v1', 'RuleTaker', 'ProofWriter', 'overall']
width = 0.2
x = np.arange(len(ds_names))

fig, ax = plt.subplots(figsize=(12, 5))

for i, method in enumerate(METHODS):
    accs = [metrics[method][ds]['overall'] for ds in ds_names]
    ax.bar(x + i * width, accs, width, label=method.upper())

ax.set_xlabel('Dataset')
ax.set_ylabel('Accuracy')
ax.set_title('RFDE vs. Baselines: Accuracy by Dataset')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(ds_names)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Visualization complete.")

## Summary

### Key Findings

1. **Accuracy**: CoT achieves highest accuracy (57.5%), followed by RFDE (51.5%), LINC (48.5%), RAG (39.6%).
2. **Significance**: McNemar test shows RFDE significantly better than CoT on RuleTaker (p=0.0077), but no overall significance (p=0.50 vs LINC, p=0.12 vs CoT).
3. **Hallucination Tradeoff**: RFDE's atomic decomposition decomposes facts to verifiable predicates, which independent verifiers flag as unsupported in natural-language documents (75% hallu rate). LINC's higher-level translation aligns better with NL (0% hallu).
4. **Cost**: $1.142 total (2017 LLM calls via Haiku 4.5), demonstrating feasibility on commodity budgets.

### Reproducibility

All 2017 LLM calls logged to `llm_calls.jsonl` with model, tokens, and cost per call. Stratified sampling ensures class-balanced benchmarks across all three datasets.

In [ ]:
# Final summary statistics
print("\n" + "="*60)
print("RFDE vs. LINC: Demand-Driven Extraction Experiment")
print("="*60)
print(f"\nExperiment Config:")
print(f"  Model: {MODEL}")
print(f"  Seed: {SEED}")
print(f"  Total examples: {TOTAL_TASKS}")
print(f"  Total cost: ${TOTAL_COST_USD:.3f}")
print(f"  Methods: {', '.join(m.upper() for m in METHODS)}")

print(f"\nOverall Accuracy:")
for method in METHODS:
    acc = metrics[method]['overall']['overall']
    hallu = hallu_rates[method]
    print(f"  {method.upper():6s}: {acc:.1%} (hallu={hallu:.1%})")

print(f"\nRFDE Decomposition:")
print(f"  Atomic: {metrics['rfde']['overall']['atomic_decomposition_count']} examples")
print(f"  Direct query: {metrics['rfde']['overall']['direct_query_count']} examples")

print("\n" + "="*60)